# Iris classification — feature audit and test gates
A standalone practice notebook for the Mage Iris exercise (C1–C4). This version explores a feature-range audit and class-wise evaluation. It retains the classroom decision tree, seed 42, 80:20 split and accuracy gate of 0.80. The executable sections below use scikit-learn directly; the final section explains how to connect the blocks to your own Mage project and obtain actual orchestration evidence. Use this as a learning reference and implement your own assignment changes.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

SEED = 42
MIN_ACCURACY = 0.80
iris = load_iris(as_frame=True)
features = list(iris.feature_names)
labels = list(range(len(iris.target_names)))
frame = iris.frame.copy()
assert len(frame) == 150
assert set(frame.target.unique()) == set(labels)
print(f"Loaded: {len(frame)} records; {len(features)} features")
display(frame.head())

Loaded: 150 records; 4 features


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


## C4 practice — feature profile audit
The proposed transformer checks the numeric feature ranges, rejects missing and constant columns, and returns the dataframe unchanged so the next block still receives the original schema. In Mage, place this between `load_iris` and `prepare_data`.

In [2]:
def feature_profile(data):
    feature_data = data[features]
    if not all(pd.api.types.is_numeric_dtype(feature_data[col]) for col in features):
        raise ValueError("Non-numeric feature")
    if not np.isfinite(feature_data.to_numpy(dtype=float)).all():
        raise ValueError("Missing or non-finite feature")
    profile = feature_data.agg(['min', 'median', 'max']).T
    profile['span'] = profile['max'] - profile['min']
    if not profile['span'].gt(0).all():
        raise ValueError('Constant feature')
    display(profile.round(2))
    return data.copy()

profiled = feature_profile(frame)
pd.testing.assert_frame_equal(frame, profiled)
print('Feature audit: PASS; original data preserved')

bad = frame.copy()
bad[features[0]] = 1.0
try:
    feature_profile(bad)
except ValueError as exc:
    print('Intentional invalid input rejected:', exc)
else:
    raise AssertionError('Audit did not reject the constant feature')

,min,median,max,span
sepal length (cm),4.3,5.80,7.9,3.6
sepal width (cm),2.0,3.00,4.4,2.4
petal length (cm),1.0,4.35,6.9,5.9
petal width (cm),0.1,1.30,2.5,2.4


Feature audit: PASS; original data preserved
Intentional invalid input rejected: Constant feature


## C1 — preparing the classroom holdout
The split is intentionally not stratified, matching the reference configuration. No preprocessing is fitted on the holdout.

In [3]:
train, holdout = train_test_split(profiled, test_size=0.2, random_state=SEED)
assert (len(train), len(holdout)) == (120, 30)
assert set(train.index).isdisjoint(holdout.index)
print(f'Train: {len(train)} | Test: {len(holdout)}')
display(pd.DataFrame({'train':train.target.value_counts().sort_index(),
                      'test':holdout.target.value_counts().sort_index()}))

def predict_at_depth(depth):
    model = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    model.fit(train[features], train.target)
    pred = holdout[['target'] + features].copy()
    pred['predicted'] = model.predict(holdout[features])
    return model, pred

Train: 120 | Test: 30


,train,test
target,,
0,40,10
1,41,9
2,39,11


## C2 — accuracy gate and prediction export
Use a disposable directory. Remove the previous export on each iteration, run the accuracy check, and save only when it passes. This demonstrates the gate in Python, not Mage scheduling.

In [4]:
def accuracy_gate(predictions):
    assert len(predictions) == 30
    score = accuracy_score(predictions.target, predictions.predicted)
    if score < MIN_ACCURACY:
        raise ValueError(f'Accuracy {score:.4f} below {MIN_ACCURACY:.2f}')
    return score

with TemporaryDirectory(prefix='iris_practice_') as tmp:
    destination = Path(tmp) / 'iris_predictions.csv'
    experiment = []
    for depth in range(1, 6):
        destination.unlink(missing_ok=True)
        _, predictions = predict_at_depth(depth)
        score = accuracy_score(predictions.target, predictions.predicted)
        try:
            accuracy_gate(predictions)
        except ValueError:
            status = 'FAIL'
        else:
            status = 'PASS'
            predictions.to_csv(destination, index=False)
            assert len(pd.read_csv(destination)) == 30
        exists = destination.exists()
        assert exists == (status == 'PASS')
        experiment.append((depth, score, status, exists))
    depth_results = pd.DataFrame(experiment, columns=['max_depth','accuracy','gate','csv_created'])
    display(depth_results.round(4))

,max_depth,accuracy,gate,csv_created
0,1,0.6333,FAIL,False
1,2,0.9667,PASS,True
2,3,1.0000,PASS,True
3,4,1.0000,PASS,True
4,5,1.0000,PASS,True


## C3 — evaluation branch
Evaluate the depth-3 model on the held-out set. The confusion matrix uses true labels as rows and predicted labels as columns. Per-class support and F1 reveal whether a single class is being neglected; macro averages give every class equal weight.

In [5]:
model, predictions = predict_at_depth(3)
score = accuracy_gate(predictions)
cm = pd.DataFrame(confusion_matrix(predictions.target, predictions.predicted, labels=labels),
                  index=pd.Index(iris.target_names, name='actual'),
                  columns=pd.Index(iris.target_names, name='predicted'))
assert cm.to_numpy().sum() == len(holdout)
report = classification_report(predictions.target, predictions.predicted, labels=labels,
                               target_names=iris.target_names, output_dict=True, zero_division=0)
class_metrics = pd.DataFrame(report).T.loc[list(iris.target_names), ['precision','recall','f1-score','support']]
summary = pd.Series({'accuracy': score, 'macro_precision': report['macro avg']['precision'],
                     'macro_recall': report['macro avg']['recall'], 'macro_f1': report['macro avg']['f1-score']})
print('Confusion matrix')
display(cm)
print('Per-class diagnostic table')
display(class_metrics.round(3))
print('Summary metrics')
display(summary.to_frame('score').round(3))
assert class_metrics.support.sum() == 30

Confusion matrix


predicted,setosa,versicolor,virginica
actual,,,
setosa,10,0,0
versicolor,0,9,0
virginica,0,0,11


Per-class diagnostic table


,precision,recall,f1-score,support
setosa,1.0,1.0,1.0,10.0
versicolor,1.0,1.0,1.0,9.0
virginica,1.0,1.0,1.0,11.0


Summary metrics


,score
accuracy,1.0
macro_precision,1.0
macro_recall,1.0
macro_f1,1.0


## Extension — decision tree feature importance
The values are impurity-based importances for this fitted model. They are not causal explanations and can be unstable with correlated measurements.

In [6]:
importance = pd.Series(model.feature_importances_, index=features, name='importance').sort_values(ascending=False)
display(importance.to_frame().round(4))
assert np.isclose(importance.sum(), 1.0)

,importance
petal length (cm),0.9346
petal width (cm),0.0654
sepal width (cm),0.0000
sepal length (cm),0.0000


## Connecting your own Mage project
1. Use the original classroom project and its four baseline blocks. Run `mage run iris_project iris_classifier --test` in a compatible Mage environment, from the project parent directory.
2. For the baseline and C2, preserve the original metadata. Consult `mage run --help` for the installed version's `--runtime-vars` format; run each depth 1–5 and remove the previous `iris_predictions.csv` before each run. Record actual logs and check whether the CSV was created.
3. For C3, wire `train_model` to `save_results` and `evaluate_model`. Add your own class-wise diagnostics in the evaluation block if useful. Test that the confusion matrix sums to 30.
4. For C4, implement a `feature_profile` transformer using the audit idea above. In the block, use Mage's `@transformer` and `@test` decorators; make the test check finite numeric data and nonconstant feature ranges. Update both directions of the metadata graph. Test the normal frame and a deliberately invalid frame.
5. Run the final pipeline with tests, then use `mage start iris_project` and capture the real branching graph in the UI. In Colab, follow the instructor's approved tunnel procedure. Stop the tunnel after collecting your evidence.

This notebook did not execute Mage CLI, start a Mage server, or capture its UI. The direct-call results are reproducible Python experiments, not a substitute for the required Mage run.